# Summary

Test calling a deployed agent

In [1]:
import os, sys
import json
import time

from vertexai import agent_engines

import vertexai

# Import libraries from the Agent Development Kit
from google.adk.agents import Agent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])



## Retrieve a deployed agent

In [5]:
dir(agent_engines)

# get a list of agents
for agent in agent_engines.list():
    print(agent)

# Delete if needed
# agent_engines.delete(resource_name="projects/1062597788108/locations/us-central1/reasoningEngines/907343383519821824",
#                      force=True)


resource name: projects/1062597788108/locations/us-central1/reasoningEngines/8489610753035206656
resource name: projects/1062597788108/locations/us-central1/reasoningEngines/4194584083407306752
resource name: projects/1062597788108/locations/us-central1/reasoningEngines/5338815048108212224


In [3]:
os.getenv("AGENT_ENGINE_ID2")

'projects/1062597788108/locations/us-central1/reasoningEngines/4194584083407306752'

In [4]:
# dir(agent_engines)
# dir(agent_engines.AgentEngine)
# # agent_engines.AgentEngine
#
# # agent_engines.AgentEngine.list()
# agent_engines.AgentEngine.create()


In [6]:

# Retreive an ADK agent
# agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

# dir(agent_engine)
# Create a session
session = agent_engine.create_session(user_id="u_123")

# type(agent_engine)


In [7]:
type(agent_engine)
type(session)

dict

## Test the agent with one query

In [9]:
q1 = "What are LEAP exams?"
# q2 = "What are the responsibilities of the board members of a California community college?"

test_result = agent_engine.stream_query(message=q1, user_id="U_123")

events = []
for event in test_result:
    events.append(event)



In [7]:
for event in events:
    print(type(event))
    print(event.keys())
    print(event['content'].keys())
    # print(event['content']['parts'])
    # print(event['grounding_metadata'])
    print(event['author'])
    print(event['actions'])
    print()




<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
rag_assistant
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}

<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
search_assistant
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}

<class 'dict'>
dict_keys(['content', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
synthesis_agent
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}



In [10]:
# for event in events:
#     if event["author"] == "synthesis_agent":
#         for entry in event["content"]["parts"]:
#             print(entry["text"])

for event in events:
    # if event["author"] == "synthesis_agent":
    print(event["author"])
    for entry in event["content"]["parts"]:
        print(entry["text"])

    print()

rag_assistant
LEAP (Limited Examination Appointment Program) exams are alternate examinations and appointment processes used by the California Community Colleges Chancellor's Office for recruiting and hiring individuals with disabilities into State service. The Department of Rehabilitation provides LEAP certification.

search_assistant
LEAP (Limited Examination and Appointment Program) exams are part of an alternative hiring process designed to help people with disabilities obtain jobs in California State Civil Service. Instead of traditional civil service exams, LEAP allows applicants to demonstrate their competencies through on-the-job testing, also known as a Job Examination Period (JEP).

Here's a breakdown of the key aspects:

*   **Eligibility:** To be eligible for LEAP, you must be certified by the California Department of Rehabilitation (DOR) as meeting specific disability requirements.
*   **LEAP Certification:** The DOR must certify that you meet the disability requirements t

## Test the agent with multiple queries in a session

In [31]:
def pretty_print_event(event):
    """Pretty prints an event with truncation for long content."""
    if "content" not in event:
        print(f"[{event.get('author', 'unknown')}]: {event}")
        return

    author = event.get("author", "unknown")
    parts = event["content"].get("parts", [])

    for part in parts:
        if "text" in part:
            text = part["text"]
            # Truncate long text to 200 characters
            if len(text) > 200:
                text = text[:197] + "..."
            print(f"[{author}]: {text}")
        elif "functionCall" in part:
            func_call = part["functionCall"]
            print(f"[{author}]: Function call: {func_call.get('name', 'unknown')}")
            # Truncate args if too long
            args = json.dumps(func_call.get("args", {}))
            if len(args) > 100:
                args = args[:97] + "..."
            print(f"  Args: {args}")
        elif "functionResponse" in part:
            func_response = part["functionResponse"]
            print(f"[{author}]: Function response: {func_response.get('name', 'unknown')}")
            # Truncate response if too long
            response = json.dumps(func_response.get("response", {}))
            if len(response) > 100:
                response = response[:97] + "..."
            print(f"  Response: {response}")

agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
print(f"Agent ID: Id='{agent_engine.resource_name}'")

# Create a session
user_id = "u_123"
session = agent_engine.create_session(user_id=user_id)
print(f"Session created: User='{user_id}', Session='{session['id']}'")

# Create some queries
queries = [
    "Hi, how are you?",
    "What are LEAP exams?",
    "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
]

# Look for events for each query
for query in queries:
    print(f"\n[user]: {query}")
    for event in agent_engine.stream_query(
        user_id=user_id,
        session_id=session["id"],
        message=query,
    ):
        print("We got here by finding an event")
        pretty_print_event(event)
        print(session.get(session["id"]))


Agent ID: Id='projects/1062597788108/locations/us-central1/reasoningEngines/5338815048108212224'
Session created: User='u_123', Session='5108327174356598784'

[user]: Hi, how are you?
We got here by finding an event
[ask_rag_agent]: I am doing well, thank you for asking! How are you today?

None

[user]: What are LEAP exams?
We got here by finding an event
[ask_rag_agent]: LEAP exams are an alternate examination and appointment process for the recruitment and hiring of individuals with disabilities into State service. Please contact the Department of Rehabilitation f...
None

[user]: What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites
We got here by finding an event
[ask_rag_agent]: The California Community Colleges Chancellor’s Office is committed to ensuring digital accessibility for people with disabilities by continually improving the user experience and applying the relev...
None


## Look at the sessions

In [7]:
agent_engine.list_sessions(user_id=user_id)

{'sessions': [{'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '8957849324595707904',
   'app_name': '5338815048108212224',
   'last_update_time': 1746668523.502969},
  {'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '3558033371378483200',
   'app_name': '5338815048108212224',
   'last_update_time': 1746668232.507558}]}

## Create - This stuff doesn't work with the deployed agent but does work if the deployed agent is called directly from code

In [12]:

def send_query_to_agent(agent, query, user_id):
    """Sends a query to the specified agent and prints the response.

    Args:
        agent: The agent to send the query to.
        query: The query to send to the agent.

    Returns:
        A tuple containing the elapsed time (in milliseconds) and the final response from the agent.
    """

    # Create a new session - if you want to keep the history of interruction you need to move the
    # creation of the session outside of this function. Here we create a new session per query
    session = session_service.create_session(app_name=AGENT_APP_NAME,
                                             user_id=user_id,)
    # Create a content object representing the user's query
    print('\nUser Query: ', query)
    content = types.Content(role='user', parts=[types.Part(text=query)])

    # Start a timer to measure the response time
    start_time = time.time()

    # Create a runner object to manage the interaction with the agent
    runner = Runner(app_name=AGENT_APP_NAME, agent=agent, artifact_service=artifact_service, session_service=session_service)

    # Run the interaction with the agent and get a stream of events
    events = runner.run(user_id=user_id, session_id=session.id, new_message=content)

    final_response = None
    elapsed_time_ms = 0.0

    # Loop through the events returned by the runner
    for _, event in enumerate(events):

        is_final_response = event.is_final_response()

        if not event.content:
             continue

        if is_final_response:
            end_time = time.time()
            elapsed_time_ms = round((end_time - start_time) * 1000, 3)

            print("-----------------------------")
            print('>>> Inside final response <<<')
            print("-----------------------------")
            final_response = event.content.parts[0].text # Get the final response from the agent
            print(f'Agent: {event.author}')
            print(f'Response time: {elapsed_time_ms} ms\n')
            print(f'Final Response:\n{final_response}')
            print("----------------------------------------------------------\n")

    return elapsed_time_ms, final_response



In [4]:
MODEL='gemini-2.0-flash-001'

# # Create a basic agent with instructions amd greeting only
# basic_agent = Agent(model=MODEL,
#     name="agent_basic",
#     description="This agent responds to inquiries about its creation by stating it was built using the Google Agent Framework.",
#     instruction="If they ask you how you were created, tell them you were created with the Google Agent Framework.",
#     generate_content_config=types.GenerateContentConfig(temperature=0.2),
# )

############## Import the agent code from the python file used to create the deployed agent
rag_path = "ccc_chatbot/"
sys.path.insert(0, rag_path)
print(os.listdir(rag_path))
from agent import root_agent

basic_agent = root_agent

# Get the agent
# basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(basic_agent, "How are you today", user_id)


['__init__.py', '__pycache__', 'agent.py', 'prompt.py', 'sub_agents']


ImportError: cannot import name 'root_agent' from partially initialized module 'agent' (most likely due to a circular import) (/Users/stephengodfrey/Documents/Workbench/Numantic/ccc-policy_assistant/test_agent_workflow/ccc_chatbot/agent.py)

In [32]:
from google.adk.agents import SequentialAgent
from google.adk.agents import LlmAgent

basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

code_pipeline_agent = SequentialAgent(
    name="run_other_agents",
    # sub_agents=[basic_agent],
    tools=[basic_agent],
    description="Wrapper for the basic agent",
)

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(code_pipeline_agent, "How are you today", user_id)

ValidationError: 1 validation error for SequentialAgent
tools
  Extra inputs are not permitted [type=extra_forbidden, input_value=[<vertexai.agent_engines....nes/4194584083407306752], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/extra_forbidden